# 02. Analyzing an Image with Azure AI Content Understanding

This notebook uses Azure AI Content Understanding's `prebuilt-imageSearch` analyzer to analyze a screenshot (`support_ticket_portal.png`) — the same `ContentUnderstandingClient` pattern as notebook 01, applied to an image instead of a PDF. It belongs to Section 07 of this course, the document/content understanding agent section.

**Difficulty: Beginner/Intermediate**

Content Understanding's image analyzers are the multimodal counterpart to the document analyzers in notebook 01 — both are prebuilt, zero-shot models exposed through the same client and long-running-operation pattern, which is exactly the kind of "one consistent SDK shape across content types" detail AI-102 rewards recognizing.

## Prerequisites

**pip3 packages:**
```bash
pip3 install azure-ai-contentunderstanding azure-core python-dotenv
```
(Same as notebook 01 — not currently in the repo root `requirements.txt`.)

**Azure resources required:**
- The same Azure AI Foundry / Azure AI Services resource with Content Understanding enabled, used in notebook 01.

**Environment variables** (same as notebook 01 — add to root `.env` if not already present):
```
AZURE_CONTENTUNDERSTANDING_ENDPOINT=https://<your-resource>.services.ai.azure.com/
AZURE_CONTENTUNDERSTANDING_KEY=<your-content-understanding-key>
```

**Local file:** this notebook expects `support_ticket_portal.png` in the **same folder as this notebook** (`07. Section Code/`) — already present alongside the original script.

## What You'll Learn

- Reusing the same `ContentUnderstandingClient` + `begin_analyze_binary` pattern from notebook 01 for a different content type (image vs PDF)
- How `content_type` (`image/png` here vs `application/pdf` in notebook 01) tells the service how to decode the binary input
- What a `prebuilt-imageSearch` analyzer returns for a UI screenshot, as opposed to a structured document

### Step 1 — Imports, configuration, and the client

Identical setup to notebook 01: an `AzureKeyCredential` and a `ContentUnderstandingClient` pointed at the same endpoint. Content Understanding treats documents and images as different `content_type`s through the same client — there's no separate "image client" class.

💡 **Exam tip:** AI-102 tests both Azure AI Vision (Image Analysis) and Content Understanding as ways to extract information from images — know the distinction: Azure AI Vision's Image Analysis API focuses on tags, captions, OCR text, and object detection with dedicated feature flags; Content Understanding's analyzer model (used here) is oriented toward structured, schema-driven extraction that can span documents, images, audio, and video under one unified API shape.

🔄 **Alternatives:** for pure image-tagging/captioning/OCR needs without the analyzer/schema machinery, the Azure AI Vision `ImageAnalysisClient` (`analyze()` with `visual_features=[...]`) is often a simpler, more direct fit — reach for Content Understanding specifically when you want structured, schema-defined output or need to handle mixed content types uniformly.

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient

load_dotenv()

endpoint = os.getenv("AZURE_CONTENTUNDERSTANDING_ENDPOINT")
api_key = os.getenv("AZURE_CONTENTUNDERSTANDING_KEY")

client = ContentUnderstandingClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(api_key),
)


### Step 2 — Submit the image for analysis

Same LRO pattern as notebook 01: open the file in binary mode, call `begin_analyze_binary(...)`, and block on `poller.result()`. The two things that change for an image versus a PDF are `analyzer_id` (`"prebuilt-imageSearch"` here, versus `"prebuilt-invoice"`) and `content_type` (`"image/png"` versus `"application/pdf"`) — the client-side flow is otherwise unchanged.

💡 **Exam tip:** `prebuilt-imageSearch` is oriented around producing a representation of the image (and the elements within it) suitable for search/retrieval scenarios — useful context if you're using Content Understanding to make a library of UI screenshots or product photos searchable by content, distinct from `prebuilt-invoice`/`prebuilt-receipt`-style field extraction from notebook 01.

🔄 **Alternatives:** other Content Understanding image analyzer IDs may target different use cases (general description, layout of a scanned image, etc.) — always check current Microsoft documentation for the up-to-date analyzer catalog, since prebuilt analyzer names/capabilities evolve.

In [ ]:
file_path = "support_ticket_portal.png"

with open(file_path, "rb") as file:
    poller = client.begin_analyze_binary(
        analyzer_id="prebuilt-imageSearch",
        binary_input=file,
        content_type="image/png",
    )

result = poller.result()
print("Analysis completed.")


### Step 3 — Inspect the result

As in notebook 01, `result["contents"][0]` is the first analyzed content entry. Unlike notebook 01 — which separately inspected `markdown` and `fields` — this cell just prints the whole `content` object, since a `prebuilt-imageSearch` result's shape is oriented differently (more description/embedding-search-oriented) than an invoice's field-extraction shape.

💡 **Exam tip:** always inspect a new analyzer's raw output shape before writing code that assumes specific keys exist — different prebuilt analyzers (invoice vs image search vs receipt vs ID document) return different result schemas under the same `contents[0]` envelope, so field names from one analyzer don't transfer to another.

🔄 **Alternatives:** once you know this analyzer's actual result shape from a real run, replace this blanket `print(content)` with targeted field access (as in notebook 01's Step 4) for production use.

In [ ]:
content = result["contents"][0]

print("\n--- Image Analysis Result ---")
print(content)


### Step 4 — Close the client

Same cleanup step as notebook 01.

💡 **Exam tip:** consistently closing (or context-managing) SDK clients is a small but real production hygiene habit — connection pools left open across many short-lived script runs can exhaust local resources.

🔄 **Alternatives:** use `with ContentUnderstandingClient(...) as client:` to handle this automatically.

In [ ]:
client.close()


## Summary

This notebook reused the `ContentUnderstandingClient` pattern from notebook 01 to analyze an image (a support-ticket-portal screenshot) instead of a PDF, using the `prebuilt-imageSearch` analyzer and `content_type="image/png"`. The output demonstrates that Content Understanding's client-side flow — construct client, `begin_analyze_binary`, `poller.result()`, inspect `contents[0]` — stays the same across content types; only the `analyzer_id` and `content_type` arguments change.

## Try It Yourself

1. **Easy:** Run this notebook's `content` object through `json.dumps(content, indent=2)` (if it's JSON-serializable) to get a readable, structured view instead of the default `print()` repr.
2. **Intermediate:** Try a different sample image (e.g., a product photo or a scanned document image) with the same analyzer and compare what gets extracted.
3. **Advanced:** Compare `prebuilt-imageSearch`'s output for the same screenshot against Azure AI Vision's `ImageAnalysisClient.analyze(visual_features=["CAPTION", "READ", "TAGS"])` output, to build intuition for when each service is the better exam-relevant choice for a given image task.